In [1]:
import pandas as pd

In [2]:
df_promotion=pd.read_csv('promotional_table_cleaned.csv')
df_pricelist=pd.read_csv('Price_list_table_cleaned.csv')

In [3]:
df_promotion.head()

,sku,PL,season,promotion
0,G0L15B,DELLG0_SUP_INK,Q1,10% off ink
1,G0L15B,DELLG0_SUP_INK,Q2,$2 off toner
2,G0L15B,DELLG0_SUP_INK,Q3,5% off toner
3,G0L15B,DELLG0_SUP_INK,Q4,20% off ink
4,7Y4T6QB,DELL7Y_PC_Desktop,Q1,$10 off laptops


In [4]:
df_pricelist.head()

,sku,PL,MAP,LPP
0,0X749QB,DELL0X_PC_Laptop,106684,98362.648
1,75X62BB,DELL75_PC_Monitor,35611,33616.784
2,DF627B,DELLDF_SUP_TON,29914,29225.978
3,4ZB96A,HP4Z_PH_LJ,39258,36706.230
4,0R0N0QB,DELL0R_PC_Laptop,139670,128775.740


In [5]:
# left Join

df_promotion = df_promotion.merge(df_pricelist[["sku", "MAP"]],on="sku",how="left")

In [32]:
df_promotion.head(5)

,sku,PL,season,promotion,MAP,PL_CATEGORY,PROMO_CATEGORY,promotion_value,Discount (%),Discount (-),Discount value 1,Discount value 2,Discount,Discount Price
0,G0L15B,DELLG0_SUP_INK,Q1,10% off ink,9900,INK,INK,9900,10,0,990.0,0,990.0,8910.0
1,G0L15B,DELLG0_SUP_INK,Q2,$2 off toner,9900,INK,Toner,0,0,2,0.0,182,0.0,9900.0
2,G0L15B,DELLG0_SUP_INK,Q3,5% off toner,9900,INK,Toner,0,5,0,495.0,0,0.0,9900.0
3,G0L15B,DELLG0_SUP_INK,Q4,20% off ink,9900,INK,INK,9900,20,0,1980.0,0,1980.0,7920.0
4,7Y4T6QB,DELL7Y_PC_Desktop,Q1,$10 off laptops,67545,Desktop,Laptop,0,0,10,0.0,910,0.0,67545.0


In [7]:
df_promotion["MAP"].isna().sum()

np.int64(0)

In [8]:
# Category Rules

CATEGORY_RULES = {
    "INK": ["INK"],
    "Toner": ["TONER", "TON"],
    "Laptop": ["LAPTOP"],
    "Desktop": ["DESKTOP", "PC"],
    "Monitor": ["MONITOR"],
    "IJ": ["INKJET", "IJ"],
    "LJ": ["LASERJET", "LASER JET", "LJ"],
    "SJ": ["SCANJET", "SCAN JET", "SJ"],
    "Unknown": ["UNK", "UNKNOWN"]
}

In [9]:
def extract_category(text):
    text = str(text).upper()

    for category, keys in CATEGORY_RULES.items():
        for k in keys:
            if k in text:
                return category

    return "Unknown"

In [10]:
df_promotion["PL_CATEGORY"] = df_promotion["PL"].apply(extract_category)

In [11]:
df_promotion["PROMO_CATEGORY"] = df_promotion["promotion"].apply(extract_category)

In [12]:
df_promotion.head()

,sku,PL,season,promotion,MAP,PL_CATEGORY,PROMO_CATEGORY
0,G0L15B,DELLG0_SUP_INK,Q1,10% off ink,9900,INK,INK
1,G0L15B,DELLG0_SUP_INK,Q2,$2 off toner,9900,INK,Toner
2,G0L15B,DELLG0_SUP_INK,Q3,5% off toner,9900,INK,Toner
3,G0L15B,DELLG0_SUP_INK,Q4,20% off ink,9900,INK,INK
4,7Y4T6QB,DELL7Y_PC_Desktop,Q1,$10 off laptops,67545,Desktop,Laptop


In [13]:
## Applying promotion values

df_promotion["promotion_value"] = (df_promotion["MAP"].where(df_promotion["PL_CATEGORY"] == df_promotion["PROMO_CATEGORY"], 0))

In [14]:
df_promotion.head(5)

,sku,PL,season,promotion,MAP,PL_CATEGORY,PROMO_CATEGORY,promotion_value
0,G0L15B,DELLG0_SUP_INK,Q1,10% off ink,9900,INK,INK,9900
1,G0L15B,DELLG0_SUP_INK,Q2,$2 off toner,9900,INK,Toner,0
2,G0L15B,DELLG0_SUP_INK,Q3,5% off toner,9900,INK,Toner,0
3,G0L15B,DELLG0_SUP_INK,Q4,20% off ink,9900,INK,INK,9900
4,7Y4T6QB,DELL7Y_PC_Desktop,Q1,$10 off laptops,67545,Desktop,Laptop,0


In [15]:
df_promotion.promotion_value.value_counts()

promotion_value
0         274
9900        2
14364       2
6749        2
7120        2
         ... 
71225       1
38447       1
253333      1
18899       1
28300       1
Name: count, Length: 76, dtype: int64

In [16]:
# regex extraction

import re

In [17]:
# --- Percentage discount ---
df_promotion["Discount (%)"] = (
    df_promotion["promotion"]
    .str.extract(r'(\d+)\s*%')[0]   # number before %
    .fillna(0)
    .astype(int)
)

In [18]:
# --- Dollar discount ---
df_promotion["Discount (-)"] = (
    df_promotion["promotion"]
    .str.extract(r'\$(\d+)')[0]     # number after $
    .fillna(0)
    .astype(int)
)

In [19]:
df_promotion.head(5)

,sku,PL,season,promotion,MAP,PL_CATEGORY,PROMO_CATEGORY,promotion_value,Discount (%),Discount (-)
0,G0L15B,DELLG0_SUP_INK,Q1,10% off ink,9900,INK,INK,9900,10,0
1,G0L15B,DELLG0_SUP_INK,Q2,$2 off toner,9900,INK,Toner,0,0,2
2,G0L15B,DELLG0_SUP_INK,Q3,5% off toner,9900,INK,Toner,0,5,0
3,G0L15B,DELLG0_SUP_INK,Q4,20% off ink,9900,INK,INK,9900,20,0
4,7Y4T6QB,DELL7Y_PC_Desktop,Q1,$10 off laptops,67545,Desktop,Laptop,0,0,10


In [20]:
df_promotion["Discount value 1"] = df_promotion["MAP"] * (df_promotion["Discount (%)"] / 100)

In [21]:
df_promotion.head()

,sku,PL,season,promotion,MAP,PL_CATEGORY,PROMO_CATEGORY,promotion_value,Discount (%),Discount (-),Discount value 1
0,G0L15B,DELLG0_SUP_INK,Q1,10% off ink,9900,INK,INK,9900,10,0,990.0
1,G0L15B,DELLG0_SUP_INK,Q2,$2 off toner,9900,INK,Toner,0,0,2,0.0
2,G0L15B,DELLG0_SUP_INK,Q3,5% off toner,9900,INK,Toner,0,5,0,495.0
3,G0L15B,DELLG0_SUP_INK,Q4,20% off ink,9900,INK,INK,9900,20,0,1980.0
4,7Y4T6QB,DELL7Y_PC_Desktop,Q1,$10 off laptops,67545,Desktop,Laptop,0,0,10,0.0


In [22]:
df_promotion["Discount value 2"] = df_promotion["Discount (-)"] * 91

In [23]:
df_promotion.head(5)

,sku,PL,season,promotion,MAP,PL_CATEGORY,PROMO_CATEGORY,promotion_value,Discount (%),Discount (-),Discount value 1,Discount value 2
0,G0L15B,DELLG0_SUP_INK,Q1,10% off ink,9900,INK,INK,9900,10,0,990.0,0
1,G0L15B,DELLG0_SUP_INK,Q2,$2 off toner,9900,INK,Toner,0,0,2,0.0,182
2,G0L15B,DELLG0_SUP_INK,Q3,5% off toner,9900,INK,Toner,0,5,0,495.0,0
3,G0L15B,DELLG0_SUP_INK,Q4,20% off ink,9900,INK,INK,9900,20,0,1980.0,0
4,7Y4T6QB,DELL7Y_PC_Desktop,Q1,$10 off laptops,67545,Desktop,Laptop,0,0,10,0.0,910


In [24]:
df_promotion["Discount"] = ((df_promotion["Discount value 1"] + df_promotion["Discount value 2"]).where(df_promotion["promotion_value"] != 0, 0))

In [25]:
df_promotion.head(5)

,sku,PL,season,promotion,MAP,PL_CATEGORY,PROMO_CATEGORY,promotion_value,Discount (%),Discount (-),Discount value 1,Discount value 2,Discount
0,G0L15B,DELLG0_SUP_INK,Q1,10% off ink,9900,INK,INK,9900,10,0,990.0,0,990.0
1,G0L15B,DELLG0_SUP_INK,Q2,$2 off toner,9900,INK,Toner,0,0,2,0.0,182,0.0
2,G0L15B,DELLG0_SUP_INK,Q3,5% off toner,9900,INK,Toner,0,5,0,495.0,0,0.0
3,G0L15B,DELLG0_SUP_INK,Q4,20% off ink,9900,INK,INK,9900,20,0,1980.0,0,1980.0
4,7Y4T6QB,DELL7Y_PC_Desktop,Q1,$10 off laptops,67545,Desktop,Laptop,0,0,10,0.0,910,0.0


In [26]:
df_promotion["Discount Price"] = df_promotion["MAP"] - df_promotion["Discount"]

In [27]:
df_promotion.head(5)

,sku,PL,season,promotion,MAP,PL_CATEGORY,PROMO_CATEGORY,promotion_value,Discount (%),Discount (-),Discount value 1,Discount value 2,Discount,Discount Price
0,G0L15B,DELLG0_SUP_INK,Q1,10% off ink,9900,INK,INK,9900,10,0,990.0,0,990.0,8910.0
1,G0L15B,DELLG0_SUP_INK,Q2,$2 off toner,9900,INK,Toner,0,0,2,0.0,182,0.0,9900.0
2,G0L15B,DELLG0_SUP_INK,Q3,5% off toner,9900,INK,Toner,0,5,0,495.0,0,0.0,9900.0
3,G0L15B,DELLG0_SUP_INK,Q4,20% off ink,9900,INK,INK,9900,20,0,1980.0,0,1980.0,7920.0
4,7Y4T6QB,DELL7Y_PC_Desktop,Q1,$10 off laptops,67545,Desktop,Laptop,0,0,10,0.0,910,0.0,67545.0


In [28]:
final_promotion_table = df_promotion[["sku", "PL", "season", "Discount Price"]]

In [29]:
final_promotion_table.head()

,sku,PL,season,Discount Price
0,G0L15B,DELLG0_SUP_INK,Q1,8910.0
1,G0L15B,DELLG0_SUP_INK,Q2,9900.0
2,G0L15B,DELLG0_SUP_INK,Q3,9900.0
3,G0L15B,DELLG0_SUP_INK,Q4,7920.0
4,7Y4T6QB,DELL7Y_PC_Desktop,Q1,67545.0


In [30]:
final_promotion_table.rename(columns={"Discount Price": "promotion"}, inplace=True)

/var/folders/dn/fkwxfg090ns1cm1p2djbztth0000gn/T/ipykernel_54181/1898019957.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_promotion_table.rename(columns={"Discount Price": "promotion"}, inplace=True)


In [31]:
final_promotion_table.to_csv("promotion_table.csv", index=False)